# 🎓 The Master Crash Course on RAG Evaluation: Metrics, LLM-as-a-Judge & Production Observability

> **Welcome to the Definitive Masterclass on Retrieval-Augmented Generation (RAG) Evaluation.**  
> Built for AI Engineers, LLM Practitioners, and candidates preparing for **AI/LLM System Design & Evaluation Interviews**.  
> Features full native support for **OpenAI** and **OpenRouter** (including free embeddings like `nvidia/nemotron-3.5-embed-1b:free`).

---

## 📌 Executive Summary & Why RAG Evaluation is Unique

Evaluating a standard machine learning model (e.g., classification or regression) relies on closed-form metrics like Accuracy, F1-score, or MSE.  
However, **RAG systems combine two fundamentally distinct subsystems**:
1. **The Retrieval Component (Information Retrieval / IR)**: Searching and fetching the most relevant passages from an external corpus.
2. **The Generation Component (LLM / Natural Language Generation)**: Synthesizing an accurate, coherent response grounded *strictly* in the retrieved context.

Because these two components can fail independently, evaluating only the final answer masks the root cause of errors. **If your final answer is wrong, did the retriever fetch the wrong chunks (Retrieval Failure), or did the LLM hallucinate despite having the right chunks (Generation Failure)?**

```
+---------------------------------------------------------------------------------------------------+
|                                  THE RAG SYSTEM ARCHITECTURE                                      |
+---------------------------------------------------------------------------------------------------+
|                                                                                                   |
|    [ User Query ]                                                                                 |
|          |                                                                                        |
|          +---------------------------------------------------+                                    |
|          |                                                   |                                    |
|          v                                                   v                                    |
|   +---------------+                                   +---------------+                           |
|   | 1. RETRIEVER  | ---> [ Retrieved Context ]  --->  | 2. GENERATOR  | ---> [ Final Answer ]     |
|   +---------------+                                   +---------------+                           |
|          ^                                                   ^                                    |
|          |                                                   |                                    |
|  [ Vector DB / BM25 ]                                   [ LLM Model ]                             |
+---------------------------------------------------------------------------------------------------+
                                           |
                                           v
+---------------------------------------------------------------------------------------------------+
|                                    THE RAG TRIAD OF TRUTH                                         |
+---------------------------------------------------------------------------------------------------+
|                                                                                                   |
|     1. CONTEXT RELEVANCE : Is the retrieved context relevant to the user query?                   |
|                            (Evaluates: Query <---> Retrieved Context)                             |
|                                                                                                   |
|     2. GROUNDEDNESS        : Is the generated answer factually grounded in the context?           |
|        / FAITHFULNESS       (Evaluates: Retrieved Context <---> Generated Answer)                 |
|                                                                                                   |
|     3. ANSWER RELEVANCE  : Does the generated answer directly fulfill the user query?             |
|                            (Evaluates: Query <---> Generated Answer)                              |
|                                                                                                   |
|     4. ANSWER CORRECTNESS: Does the generated answer match the Ground-Truth reference?            |
|                            (Evaluates: Ground Truth <---> Generated Answer)                       |
+---------------------------------------------------------------------------------------------------+
```

---

## 📊 Comprehensive Metric Taxonomical Cheat Sheet

| Metric Category | Metrics Included | Target Component | Reference Needed? | Primary Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Traditional Lexical** | Exact Match, BLEU-1/4, ROUGE-1/2/L, Token F1 | Final Generator Output | **Yes** (Ground-Truth Answer) | Fast smoke-tests, regression testing against static baselines |
| **Information Retrieval (IR)** | Hit Rate @ K, MRR @ K, Precision @ K, NDCG @ K | Vector DB / Retriever | **Yes** (Ground-Truth Contexts) | Tuning chunk size, overlap, embedding models, and Top-K |
| **Semantic Similarity** | Embedding Cosine Similarity, Cross-Encoder BERTScore | Final Generator Output | **Yes** (Ground-Truth Answer) | Capturing semantic equivalence beyond exact word overlap |
| **RAG Triad (LLM-as-Judge)** | Faithfulness (Groundedness), Answer Relevance, Context Relevance | Full RAG Pipeline | **NO (Reference-Free!)** | **Production Online Monitoring**, Hallucination Detection |
| **End-to-End LLM-as-Judge** | Answer Correctness, Negative Constraint Checking, Aspect Critique | Full RAG Pipeline | **Yes** (Reference Answer) | Offline benchmarking, pre-release model comparisons |

---

## 🗺️ Masterclass Curriculum Roadmap

1. **Module 1**: Environment Configuration & Multi-Provider Observability (OpenRouter & OpenAI + LangSmith).
2. **Module 2**: The Baseline – Why Classical Lexical Metrics (BLEU/ROUGE) Fail for Generative AI (Live Demos).
3. **Module 3**: Information Retrieval (IR) & Ranking Metrics – Evaluating the Retriever Alone (Hit Rate@K, MRR@K).
4. **Module 4**: Building the Production-Grade RAG Pipeline with OpenRouter Free Embeddings (`nvidia/nemotron-3.5-embed-1b:free`).
5. **Module 5**: Curating Golden Evaluation Datasets (Factoids, Multi-Hop, and Unanswerable Negative Controls).
6. **Module 6**: The RAG Triad & Structured LLM-as-a-Judge Evaluators (Chain-of-Thought Explanation-First Schemas).
7. **Module 7**: Advanced LLM-as-a-Judge Paradigms & Bias Mitigation (Position, Verbosity, and Self-Enhancement Biases).
8. **Module 8**: Leading Industry Frameworks Comparison (Ragas vs TruLens vs LangSmith vs DeepEval).
9. **Module 9**: Running the Full Benchmark Suite & Building a Local Pandas Analytics Dashboard.
10. **Module 10**: The AI Engineer Interview Preparation Master Guide (10 High-Frequency Questions & Answers).


---
# 🧩 Module 1: Environment Setup & Production Observability

### 🔍 What is happening here?
We initialize our runtime environment, load API credentials from `.env`, and configure **LangSmith** distributed tracing alongside our model provider (**OpenRouter** or **OpenAI**).

### 💡 Why do we do it?
In enterprise AI applications, you cannot debug or optimize what you cannot measure. LangSmith provides **distributed tracing for LLM pipelines**:
- Inspecting every retrieval step (chunks retrieved, similarity scores, latency).
- Tracking token consumption and financial cost per call.
- Capturing intermediate execution graphs for root-cause error analysis.
- Serving as the centralized evaluation platform for offline regression testing.

### 🌐 OpenRouter vs OpenAI Configuration:
This notebook auto-detects `OPENROUTER_API_KEY` or `OPENAI_API_KEY`.
- If using OpenRouter, we can leverage **completely free embedding models** like `nvidia/nemotron-3.5-embed-1b:free` alongside efficient LLMs like `openai/gpt-4o-mini`.

### 🎯 Interview Insight:
> **Question**: *Why is distributed tracing critical when building and evaluating production RAG systems?*  
> **Answer**: A RAG pipeline is a multi-step distributed graph (Embedding -> Vector Search -> Context Serialization -> Prompt Assembly -> LLM Completion). Without tracing, a latency spike or hallucination cannot be localized. Tracing provides atomic span-level observability for retrieval latency, token costs, chunk relevance, and LLM reasoning steps.


In [30]:
import os
from dotenv import load_dotenv

# Use override=True so changes in .env are always picked up immediately
load_dotenv(override=True)

# Set User-Agent to suppress web loader warnings
os.environ["USER_AGENT"] = "RAG-Evaluation-Masterclass/1.0"

# Verify and set essential API keys
langsmith_key = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
if langsmith_key:
    os.environ["LANGSMITH_API_KEY"] = langsmith_key
    os.environ["LANGCHAIN_API_KEY"] = langsmith_key
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "RAG-Master-Evaluation")
    tracing_status = "ENABLED ✅"
else:
    # Crucial: Disable tracing if key is absent or empty to prevent 401 Unauthorized errors!
    os.environ["LANGSMITH_TRACING"] = "false"
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    tracing_status = "DISABLED (Set LANGSMITH_API_KEY in .env to enable)"

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY", "")

# Provider selection logic
USE_OPENROUTER = bool(os.getenv("OPENROUTER_API_KEY"))

print("✅ Environment successfully configured.")
print(f"📊 LangSmith Tracing: {tracing_status}")
print(f"🌐 Active Model Provider: {'OpenRouter (Free tier / Multi-model)' if USE_OPENROUTER else 'OpenAI Direct'}")
print(f"🔑 OpenRouter Key: {'Configured ✅' if os.getenv('OPENROUTER_API_KEY') else 'Not Found'}")
print(f"🔑 OpenAI Key: {'Configured ✅' if os.getenv('OPENAI_API_KEY') else 'Not Found'}")


✅ Environment successfully configured.
📊 LangSmith Tracing: ENABLED ✅
🌐 Active Model Provider: OpenRouter (Free tier / Multi-model)
🔑 OpenRouter Key: Configured ✅
🔑 OpenAI Key: Configured ✅


---
# ⚖️ Module 2: Classical NLP Metrics vs LLM Evaluation (The Paradigm Shift)

### 🔍 What is happening here?
Before LLM-as-a-judge existed, NLP systems were evaluated using **lexical n-gram overlap metrics**:
1. **Exact Match (EM)**: Binary 1 if candidate text is identical character-for-character to reference, else 0.
2. **BLEU (Bilingual Evaluation Understudy)**: Measures modified n-gram precision with a brevity penalty. Originally built for machine translation.
3. **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)**:
   - **ROUGE-1**: Unigram (single word) overlap recall/F1.
   - **ROUGE-2**: Bigram (two-word phrase) overlap recall/F1.
   - **ROUGE-L**: Longest Common Subsequence (LCS) overlap, accounting for sentence-level word order.
4. **Token F1**: Harmonic mean of token-level precision and recall.

### 💡 Why do classical metrics FAIL for generative RAG?
We will run a live empirical test with two classic failure cases:
1. **Failure Case 1 (Paraphrasing / Semantic Equivalence)**: Two sentences mean the exact same thing, but use different vocabulary. Classical metrics score them near zero!
2. **Failure Case 2 (Catastrophic Negation / Hallucination)**: A single inserted word ("not") completely flips the medical meaning of an answer. Classical metrics award it an **85%+ F1 score**!

### 🎯 Interview Insight:
> **Question**: *Can we use BLEU or ROUGE to evaluate a production RAG application? Explain why or why not.*  
> **Answer**: No. BLEU and ROUGE are purely surface-level lexical matchers. They are blind to semantics, synonyms, and paraphrasing. More critically, they are blind to negation and factual truth: an answer that introduces a fatal hallucination while sharing 90% of the reference words will receive a near-perfect ROUGE score, while a creative, 100% accurate paraphrase will receive a near-zero score.


In [31]:
import re
from collections import Counter

def tokenize(text: str) -> list[str]:
    """Simple word tokenizer converting text to lowercase alphanumeric tokens."""
    return re.findall(r"\w+", text.lower())

def compute_exact_match(prediction: str, reference: str) -> float:
    """Exact Match (EM): 1.0 if normalized strings are identical, else 0.0."""
    return float(prediction.strip().lower() == reference.strip().lower())

def compute_token_f1(prediction: str, reference: str) -> float:
    """Calculates token-level F1 score (Harmonic mean of precision and recall)."""
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)
    
    if not pred_tokens or not ref_tokens:
        return 0.0
    
    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)
    return (2 * precision * recall) / (precision + recall)

def compute_rouge_1_recall(prediction: str, reference: str) -> float:
    """Calculates unigram ROUGE-1 recall."""
    pred_tokens = set(tokenize(prediction))
    ref_tokens = set(tokenize(reference))
    if not ref_tokens:
        return 0.0
    return len(pred_tokens & ref_tokens) / len(ref_tokens)

# ==============================================================================
# LIVE TEST: Demonstrating Classical Metric Pitfalls
# ==============================================================================

# Case 1: Perfect semantic equivalent, but different words
ref_1 = "Paris is the capital city of France."
pred_1 = "The French administrative capital is Paris."

# Case 2: Fatal hallucination (negation inserted), but high word overlap
ref_2 = "The biopsy confirmed the patient does not have malignant lymphoma."
pred_2 = "The biopsy confirmed the patient does have malignant lymphoma."

print("================================================================================")
print("EXPERIMENT 1: Perfect Paraphrase (Semantically identical, lexically different)")
print("Reference :", ref_1)
print("Prediction:", pred_1)
print(f"-> Exact Match Score: {compute_exact_match(pred_1, ref_1):.2f}")
print(f"-> Token F1 Score   : {compute_token_f1(pred_1, ref_1):.2f} (Penalizes valid paraphrasing!)")
print(f"-> ROUGE-1 Recall   : {compute_rouge_1_recall(pred_1, ref_1):.2f}")

print("\n================================================================================")
print("EXPERIMENT 2: Catastrophic Factual Reversal (Only 1 word changed: 'not' deleted)")
print("Reference :", ref_2)
print("Prediction:", pred_2)
print(f"-> Exact Match Score: {compute_exact_match(pred_2, ref_2):.2f}")
print(f"-> Token F1 Score   : {compute_token_f1(pred_2, ref_2):.2f}  <-- DANGEROUS! Scores 95% despite fatal hallucination!")
print(f"-> ROUGE-1 Recall   : {compute_rouge_1_recall(pred_2, ref_2):.2f}")
print("================================================================================")


EXPERIMENT 1: Perfect Paraphrase (Semantically identical, lexically different)
Reference : Paris is the capital city of France.
Prediction: The French administrative capital is Paris.
-> Exact Match Score: 0.00
-> Token F1 Score   : 0.62 (Penalizes valid paraphrasing!)
-> ROUGE-1 Recall   : 0.57

EXPERIMENT 2: Catastrophic Factual Reversal (Only 1 word changed: 'not' deleted)
Reference : The biopsy confirmed the patient does not have malignant lymphoma.
Prediction: The biopsy confirmed the patient does have malignant lymphoma.
-> Exact Match Score: 0.00
-> Token F1 Score   : 0.95  <-- DANGEROUS! Scores 95% despite fatal hallucination!
-> ROUGE-1 Recall   : 0.89


---
# 🎯 Module 3: Information Retrieval (IR) & Ranking Metrics (Evaluating the Retriever Alone)

### 🔍 What is happening here?
Before asking the LLM to generate an answer, we must verify that the **Retriever is doing its job**.  
If the vector database retrieves irrelevant or missing chunks, the LLM is doomed to either refuse or hallucinate. This is known as **"Garbage In, Garbage Out"**.

### 💡 The Mathematical Formulations of Retrieval Metrics:

#### 1. Hit Rate @ K (Recall @ K)
The percentage of queries for which at least one relevant chunk appears within the top-$K$ retrieved results:
$$\text{Hit Rate@K} = \frac{1}{|Q|} \sum_{q \in Q} \mathbb{I}(\text{Gold Context} \in \text{Top-K}(q))$$

#### 2. Mean Reciprocal Rank (MRR @ K)
Measures the rank position of the **very first relevant document**. Higher is better (1.0 means the relevant document was at Rank 1):
$$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$
*(If no relevant chunk is retrieved within top-$K$, reciprocal rank is 0).*

#### 3. Precision @ K
Proportion of the top-$K$ retrieved chunks that are genuinely relevant to the query:
$$\text{Precision@K} = \frac{|\text{Relevant Chunks in Top-K}|}{K}$$

### 🎯 Interview Insight:
> **Question**: *Suppose your RAG pipeline has a Hit Rate@5 of 92%, but an MRR@5 of only 0.28. What does this mean, and what engineering change would you make?*  
> **Answer**:  
> - A Hit Rate of 92% means the relevant chunk is almost always present in the top-5 retrieved chunks.  
> - However, an MRR of 0.28 means the relevant chunk is consistently buried at Rank 4 or Rank 5 ($1/4 = 0.25$, $1/3.5 \approx 0.28$).  
> - **The Problem**: LLMs suffer from the **"Lost in the Middle"** phenomenon; when relevant info is at the end or middle of long prompts, attention degradation occurs.  
> - **The Fix**: Introduce a **Cross-Encoder Re-ranker** (e.g., Cohere Rerank, BGE-Reranker) on top of the initial vector search to re-order the top-5 candidates so the most relevant chunk lands at Rank 1!


In [32]:
def calculate_hit_rate(retrieved_ids: list[str], gold_ids: set[str], k: int) -> float:
    """Calculates Hit Rate @ K: 1.0 if any gold ID is in retrieved[:k], else 0.0."""
    top_k = set(retrieved_ids[:k])
    return 1.0 if bool(top_k & gold_ids) else 0.0

def calculate_reciprocal_rank(retrieved_ids: list[str], gold_ids: set[str], k: int) -> float:
    """Calculates Reciprocal Rank @ K for a single query (1 / rank of first hit)."""
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in gold_ids:
            return 1.0 / rank
    return 0.0

def calculate_precision_at_k(retrieved_ids: list[str], gold_ids: set[str], k: int) -> float:
    """Calculates Precision @ K (fraction of top-k docs that are relevant)."""
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in top_k if doc_id in gold_ids)
    return hits / k

# ==============================================================================
# SIMULATION: Evaluating a Baseline Retriever vs. Re-Ranked Retriever
# ==============================================================================

baseline_results = [
    {"retrieved": ["doc_A", "doc_B", "doc_C", "doc_GOLD1", "doc_E"], "gold": {"doc_GOLD1"}}, # Gold at Rank 4
    {"retrieved": ["doc_F", "doc_G", "doc_H", "doc_I", "doc_GOLD2"], "gold": {"doc_GOLD2"}}, # Gold at Rank 5
    {"retrieved": ["doc_J", "doc_K", "doc_L", "doc_M", "doc_N"],     "gold": {"doc_GOLD3"}}, # Gold not in top-5
]

reranked_results = [
    {"retrieved": ["doc_GOLD1", "doc_A", "doc_B", "doc_C", "doc_E"], "gold": {"doc_GOLD1"}}, # Re-ranker brought to Rank 1!
    {"retrieved": ["doc_GOLD2", "doc_F", "doc_G", "doc_H", "doc_I"], "gold": {"doc_GOLD2"}}, # Re-ranker brought to Rank 1!
    {"retrieved": ["doc_J", "doc_K", "doc_L", "doc_M", "doc_N"],     "gold": {"doc_GOLD3"}}, # Still missed
]

def evaluate_retrieval_benchmark(data, k=5):
    hit_rates = [calculate_hit_rate(item["retrieved"], item["gold"], k) for item in data]
    mrrs = [calculate_reciprocal_rank(item["retrieved"], item["gold"], k) for item in data]
    precisions = [calculate_precision_at_k(item["retrieved"], item["gold"], k) for item in data]
    return {
        f"Hit Rate @ {k}": sum(hit_rates) / len(hit_rates),
        f"MRR @ {k}": sum(mrrs) / len(mrrs),
        f"Precision @ {k}": sum(precisions) / len(precisions)
    }

print("📊 BASELINE VECTOR RETRIEVER METRICS:")
for metric, val in evaluate_retrieval_benchmark(baseline_results, k=5).items():
    print(f"  {metric:<18}: {val:.3f}")

print("\n🚀 AFTER RE-RANKING WITH CROSS-ENCODER:")
for metric, val in evaluate_retrieval_benchmark(reranked_results, k=5).items():
    print(f"  {metric:<18}: {val:.3f}")


📊 BASELINE VECTOR RETRIEVER METRICS:
  Hit Rate @ 5      : 0.667
  MRR @ 5           : 0.150
  Precision @ 5     : 0.133

🚀 AFTER RE-RANKING WITH CROSS-ENCODER:
  Hit Rate @ 5      : 0.667
  MRR @ 5           : 0.667
  Precision @ 5     : 0.133


---
# 🏗️ Module 4: Building the Target RAG Application Pipeline (with OpenRouter Free Embeddings)

### 🔍 What is happening here?
We build a full, production-style RAG pipeline with flexible support for **OpenRouter** and **OpenAI**:
1. **OpenRouter Embeddings (`nvidia/nemotron-3.5-embed-1b:free`)**:
   - High-performance, **2048-dimensional** dense embedding model provided for **free** on OpenRouter.
   - We demonstrate both:
     - **Method A**: The direct OpenAI SDK invocation with `client.embeddings.create`.
     - **Method B**: LangChain's `OpenAIEmbeddings` wrapper with `openai_api_base="https://openrouter.ai/api/v1"`.
2. **Document Loading**: Ingesting authoritative technical posts by Lilian Weng (OpenAI researcher) on Autonomous Agents, Prompt Engineering, and Adversarial Attacks.
3. **Chunking Strategy**: Using `RecursiveCharacterTextSplitter.from_tiktoken_encoder` with `chunk_size=250` tokens and `chunk_overlap=30` tokens.
4. **Vector Store**: Initializing an in-memory vector store (`InMemoryVectorStore`).
5. **Traceable RAG Chain**: Implementing `@traceable` from `langsmith` so that every execution automatically logs the input question, retrieved documents, prompt template, and final LLM response to LangSmith.

### 💡 Why do we pass `check_embedding_ctx_length=False` in LangChain?
By default, LangChain's `OpenAIEmbeddings` attempts to validate context length locally using OpenAI's `tiktoken` tokenizer. Because `nvidia/nemotron-3.5-embed-1b:free` uses NVIDIA/Llama-style tokenization, setting `check_embedding_ctx_length=False` instructs LangChain to send texts directly to OpenRouter without local tokenizer mismatch errors!


In [40]:
import os
from dotenv import load_dotenv

# 1. Use override=True so changes in .env are always refreshed in Jupyter
load_dotenv(override=True)

# 2. Suppress WebLoader user-agent warnings
os.environ["USER_AGENT"] = "RAG-Evaluation-Masterclass/1.0"

# 3. Synchronize keys and disable tracing if key is missing (prevents 401 spam)
langsmith_key = os.getenv("LANGSMITH_API_KEY") or os.getenv(
    "LANGCHAIN_API_KEY"
)
if langsmith_key:
  os.environ["LANGSMITH_API_KEY"] = langsmith_key
  os.environ["LANGCHAIN_API_KEY"] = langsmith_key
  os.environ["LANGSMITH_TRACING"] = "true"
  os.environ["LANGCHAIN_TRACING_V2"] = "true"
  os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
else:
  os.environ["LANGSMITH_TRACING"] = "false"
  os.environ["LANGCHAIN_TRACING_V2"] = "false"


In [41]:
from openai import OpenAI

# ==============================================================================
# DEMONSTRATION: Raw OpenRouter Embedding Creation (NVIDIA Nemotron 3.5 1B Free)
# ==============================================================================
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key:
    # Initialize standard OpenAI SDK client configured for OpenRouter
    openrouter_client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=openrouter_api_key,
    )

    # Single or Batch embedding call
    sample_text = "Autonomous agents integrate reasoning, planning, and memory."
    raw_embedding = openrouter_client.embeddings.create(
        extra_headers={
            "HTTP-Referer": "https://localhost:8000",
            "X-OpenRouter-Title": "RAG Masterclass Evaluation",
        },
        model="nvidia/nemotron-3.5-embed-1b:free",
        input=sample_text,
        encoding_format="float"
    )

    print("⚡ OPENROUTER EMBEDDING DEMO:")
    print(f"Model: nvidia/nemotron-3.5-embed-1b:free")
    print(f"Embedding Vector Dimensions: {len(raw_embedding.data[0].embedding)}")
    print(f"Sample First 5 Floats: {raw_embedding.data[0].embedding[:5]}")
else:
    print("ℹ️ Note: OPENROUTER_API_KEY not found in .env, skipping standalone OpenRouter test.")


⚡ OPENROUTER EMBEDDING DEMO:
Model: nvidia/nemotron-3.5-embed-1b:free
Embedding Vector Dimensions: 2048
Sample First 5 Floats: [0.11077881, -0.04107666, -0.01574707, -0.0019512177, -0.012649536]


In [42]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import traceable

# 1. Source URLs from technical blogs and docs (requires 'beautifulsoup4' installed)
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://openrouter.ai/docs/quickstart",
    "https://docs.litellm.ai/docs/proxy/docker_quick_start",
]

print("📥 Loading source documents...")
raw_docs = []
for url in urls:
    try:
        loader = WebBaseLoader(url)
        raw_docs.extend(loader.load())
    except Exception as e:
        print(f"⚠️ Warning loading {url}: {e}")

print(f"📄 Loaded {len(raw_docs)} parent documents.")

# 2. Text Splitting with Tiktoken token-aware chunking
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=30
)
doc_splits = text_splitter.split_documents(raw_docs)
print(f"✂️ Created {len(doc_splits)} document chunks.")

# 3. Dynamic Vector Store & Embedding Initialization (OpenRouter or OpenAI)
if os.getenv("OPENROUTER_API_KEY"):
    print("🌐 Configuring Embeddings via OpenRouter (nvidia/nemotron-3.5-embed-1b:free)...")
    embeddings = OpenAIEmbeddings(
        openai_api_base="https://openrouter.ai/api/v1",
        openai_api_key=os.getenv("OPENROUTER_API_KEY"),
        model="nvidia/nemotron-3.5-embed-1b:free",
        check_embedding_ctx_length=False  # Bypass local tiktoken validation for NVIDIA tokenizer
    )
    generator_llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model="openai/gpt-4o-mini",
        temperature=0
    )
else:
    print("🌐 Configuring Embeddings via OpenAI direct (text-embedding-3-small)...")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    generator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 4. Full RAG Application with LangSmith Tracing
@traceable(name="rag_application_bot")
def rag_bot(question: str) -> dict:
    """Executes end-to-end RAG: retrieves relevant chunks, builds prompt, and generates grounded answer."""
    retrieved_docs = retriever.invoke(question)
    docs_context = "\n\n".join(f"[Doc {i+1}]: {doc.page_content}" for i, doc in enumerate(retrieved_docs))

    system_instructions = (
        "You are an expert AI research assistant. Use ONLY the provided source documents to answer the user question. "
        "Rules to follow:\n"
        "1. If the source documents do not contain sufficient information to answer the question, state: "
        "'I do not have sufficient information in the provided context to answer this question.'\n"
        "2. Do not hallucinate or extrapolate beyond the provided text.\n"
        "3. Keep your response concise, professional, and factual (3-4 sentences maximum)."
    )
    
    user_prompt = f"Source Documents:\n{docs_context}\n\nUser Question: {question}"

    response = generator_llm.invoke([
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_prompt}
    ])
    
    return {
        "answer": response.content,
        "documents": retrieved_docs
    }

# Sanity check
test_run = rag_bot("What are the key components of an LLM-powered autonomous agent system?")
print("\n🤖 SANITY CHECK TEST RESULT:")
print("Question: What are the key components of an LLM-powered autonomous agent system?")
print("Answer  :", test_run["answer"])
print(f"Retrieved Documents Count: {len(test_run['documents'])}")


📥 Loading source documents...
⚠️ Warning loading https://lilianweng.github.io/posts/2023-06-23-agent/: No module named 'bs4'
⚠️ Warning loading https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/: No module named 'bs4'
⚠️ Warning loading https://openrouter.ai/docs/quickstart: No module named 'bs4'
⚠️ Warning loading https://docs.litellm.ai/docs/proxy/docker_quick_start: No module named 'bs4'
📄 Loaded 0 parent documents.
✂️ Created 0 document chunks.
🌐 Configuring Embeddings via OpenRouter (nvidia/nemotron-3.5-embed-1b:free)...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a18904-db2e-77f0-a14c-a40ba76fa33c,id=01a18904-db2e-77f0-a14c-a40ba76fa33c; trace=01a18904-db2e-77f0-a14c-a40ba76fa33c,id=01a18904-db5c-7923-aead-15e41bf80882
Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a18904-db2e-77f0-a14c-a40ba76fa33c,id=01a18904-db5c-7923-aead-15e41bf80882; trace=01a18904-db2e-77f0-a14c-a40ba76fa33c,id=01a18904-de42-7512-baba-7148bfb1df97



🤖 SANITY CHECK TEST RESULT:
Question: What are the key components of an LLM-powered autonomous agent system?
Answer  : I do not have sufficient information in the provided context to answer this question.
Retrieved Documents Count: 0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a18904-db2e-77f0-a14c-a40ba76fa33c,id=01a18904-de42-7512-baba-7148bfb1df97; trace=01a18904-db2e-77f0-a14c-a40ba76fa33c,id=01a18904-db2e-77f0-a14c-a40ba76fa33c


---
# 📋 Module 5: Designing Golden Evaluation Datasets & Negative Controls

### 🔍 What is happening here?
We design a **Golden Evaluation Dataset** and register it with the LangSmith Client.

### 💡 What makes a complete RAG evaluation dataset?
A real-world RAG test suite must contain multiple categories of queries:
1. **Direct Factoid Queries**: Testing exact factual retrieval and synthesis.
2. **Multi-Point / Multi-Hop Queries**: Testing if the model aggregates facts across separate chunks.
3. **Safety / Security Queries**: Testing adversarial and jailbreak knowledge extraction.
4. **⚠️ NEGATIVE CONTROL / UNANSWERABLE QUERIES**:
   - In production, users will ask questions that your knowledge base does not cover.
   - If you only test questions that *have* answers, you will never know if your model hallucinates when context is absent!
   - A robust RAG bot must gracefully say *"I do not have sufficient information"* rather than making up plausible fiction.

### 🎯 Interview Insight:
> **Question**: *Why is it dangerous to evaluate RAG systems only on questions whose answers exist in the corpus?*  
> **Answer**: Evaluating only answerable queries fails to measure **hallucination resistance (specificity)**. In production, out-of-domain queries inevitably arrive. If your evaluation suite lacks negative controls, you cannot measure whether the system adheres to refusal instructions when the retriever returns low-similarity noise.


In [35]:
from langsmith import Client

client = Client()

golden_examples = [
    {
        "inputs": {"question": "How does the ReAct agent framework integrate reasoning and acting?"},
        "outputs": {"answer": "ReAct integrates reasoning traces and task-specific actions in an interleaved manner. Reasoning helps the model induce, track, and update action plans, while actions allow it to interface with external tools like search APIs."},
        "metadata": {"category": "Multi-Hop Reasoning", "difficulty": "Medium"}
    },
    {
        "inputs": {"question": "What are three common types of biases that arise in few-shot prompting?"},
        "outputs": {"answer": "The three common biases in few-shot prompting are: (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
        "metadata": {"category": "Direct Factoid", "difficulty": "Easy"}
    },
    {
        "inputs": {"question": "What are five primary categories of adversarial attacks on LLMs?"},
        "outputs": {"answer": "Five categories of adversarial attacks include token manipulation, gradient-based attacks, jailbreak prompting, human red-teaming, and model red-teaming."},
        "metadata": {"category": "Taxonomy / Security", "difficulty": "Medium"}
    },
    {
        "inputs": {"question": "What role does Self-Reflection play in autonomous agent architectures?"},
        "outputs": {"answer": "Self-reflection allows autonomous agents to iteratively refine past action trajectories, learn from previous trial errors, and update internal memory for future decision-making."},
        "metadata": {"category": "Synthesis", "difficulty": "Hard"}
    },
    # Negative Control Query (Context lacks information!)
    {
        "inputs": {"question": "What is the quantum superposition frequency used by Lilian Weng to build teleportation robots?"},
        "outputs": {"answer": "I do not have sufficient information in the provided context to answer this question."},
        "metadata": {"category": "Negative Control / Refusal Test", "difficulty": "Critical"}
    }
]

dataset_name = "RAG Masterclass Golden Evaluation Dataset"

existing_datasets = [d.name for d in client.list_datasets()]
if dataset_name in existing_datasets:
    print(f"Dataset '{dataset_name}' already exists. Reusing existing dataset.")
else:
    created_dataset = client.create_dataset(dataset_name=dataset_name, description="Master RAG Evaluation Suite with Negative Controls")
    client.create_examples(
        dataset_id=created_dataset.id,
        examples=golden_examples
    )
    print(f"✨ Successfully created LangSmith dataset: '{dataset_name}' with {len(golden_examples)} curated test cases.")


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a18903-6704-7633-bc2f-9a961af7fa5d,id=01a18903-69d8-7482-817a-d5e7a41b5db1; trace=01a18903-6704-7633-bc2f-9a961af7fa5d,id=01a18903-6704-7633-bc2f-9a961af7fa5d


✨ Successfully created LangSmith dataset: 'RAG Masterclass Golden Evaluation Dataset' with 5 curated test cases.


---
# ⚖️ Module 6: The RAG Triad & Structured LLM-as-a-Judge Evaluators

### 🔍 What is happening here?
We implement the **4 Pillars of RAG Evaluation** using modern **LLM-as-a-Judge** with **Structured Outputs (`with_structured_output`)**:
1. **Evaluator 1: Groundedness / Faithfulness (Response vs Retrieved Docs)**:
   - *Question asked*: "Is every claim in the generated answer directly supported by the retrieved facts?"
   - *Detects*: Hallucinations and extraneous assumptions.
2. **Evaluator 2: Answer Relevance (Response vs User Question)**:
   - *Question asked*: "Does the generated answer directly address the user's specific question?"
   - *Detects*: Wandering, generic evasiveness, or irrelevant tangents.
3. **Evaluator 3: Context Relevance / Retrieval Precision (Retrieved Docs vs User Question)**:
   - *Question asked*: "Do the retrieved documents contain information pertinent to the query, or are they noisy distractions?"
   - *Detects*: Retrieval noise, low-quality chunking, embedding drift.
4. **Evaluator 4: Answer Correctness (Response vs Ground-Truth Reference)**:
   - *Question asked*: "Is the generated answer factually accurate compared to the human reference answer?"
   - *Detects*: Semantic inaccuracy and omission of critical facts.
5. **Evaluator 5: Concision & Refusal Adherence**:
   - Verifies that unanswerable queries are properly rejected and responses remain concise.

### 💡 Why Explanation-First Reasoning (Chain-of-Thought) is Mandatory:
Notice how every `TypedDict` schema below places `explanation: str` **BEFORE** the `score` or `boolean`:
```python
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning step-by-step FIRST"]
    grounded: Annotated[bool, ..., "Final verdict"]
```
Because autoregressive language models generate tokens sequentially, **generating the explanation first forces the model to deliberate on the facts before committing to a score**. If the score is generated first, the model commits to a premature verdict and then rationalizes its mistakes!


In [36]:
from typing_extensions import Annotated, TypedDict

# Initialize dedicated evaluation judges with temperature=0 for determinism
if os.getenv("OPENROUTER_API_KEY"):
    judge_llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model="openai/gpt-4o-mini",
        temperature=0
    )
else:
    judge_llm = ChatOpenAI(model="gpt-4o", temperature=0)

# ==============================================================================
# 1. FAITHFULNESS / GROUNDEDNESS EVALUATOR (Hallucination Detection)
# ==============================================================================
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Provide a step-by-step breakdown analyzing whether each sentence in the student answer is supported by the context."]
    grounded: Annotated[bool, ..., "True if ALL claims in the answer are supported by the facts; False if ANY hallucination exists."]

grounded_prompt = """You are a strict academic judge evaluating factual groundedness.
You are given:
1. FACTS (Retrieved Context)
2. STUDENT ANSWER (Model Generated Response)

Grading Criteria:
- Verify that every factual claim in the STUDENT ANSWER can be directly deduced from the FACTS.
- If the STUDENT ANSWER contains information NOT present in the FACTS, it is a hallucination -> Mark False.
- If the STUDENT ANSWER states that the facts do not contain enough information to answer, and that is true -> Mark True.
- Provide your explanation first, then emit the final grounded verdict."""

grounded_judge = judge_llm.with_structured_output(GroundedGrade, method="json_schema", strict=True)

def eval_groundedness(inputs: dict, outputs: dict) -> dict:
    """Evaluates if the response is strictly grounded in the retrieved documents."""
    docs_text = "\n\n".join(doc.page_content for doc in outputs.get("documents", []))
    prompt = f"FACTS:\n{docs_text}\n\nSTUDENT ANSWER:\n{outputs.get('answer', '')}"
    result = grounded_judge.invoke([
        {"role": "system", "content": grounded_prompt},
        {"role": "user", "content": prompt}
    ])
    return {"key": "groundedness", "score": int(result["grounded"]), "comment": result["explanation"]}

# ==============================================================================
# 2. ANSWER RELEVANCE EVALUATOR (Query Fulfillment)
# ==============================================================================
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain whether the student answer directly addresses the specific user question asked."]
    relevant: Annotated[bool, ..., "True if the answer directly answers the question; False if it evades or goes off-topic."]

relevance_prompt = """You are an expert examiner grading answer relevance.
You are given:
1. QUESTION
2. STUDENT ANSWER

Grading Criteria:
- Assess whether the STUDENT ANSWER directly and effectively answers the QUESTION.
- Penalize answers that provide irrelevant tangents, evade the question, or fail to address the core prompt.
- If the question cannot be answered from context and the student appropriately refuses, that counts as relevant (it accurately addresses the prompt's answerability)."""

relevance_judge = judge_llm.with_structured_output(RelevanceGrade, method="json_schema", strict=True)

def eval_answer_relevance(inputs: dict, outputs: dict) -> dict:
    """Evaluates if the generated answer directly answers the user's question."""
    prompt = f"QUESTION:\n{inputs.get('question', '')}\n\nSTUDENT ANSWER:\n{outputs.get('answer', '')}"
    result = relevance_judge.invoke([
        {"role": "system", "content": relevance_prompt},
        {"role": "user", "content": prompt}
    ])
    return {"key": "answer_relevance", "score": int(result["relevant"]), "comment": result["explanation"]}

# ==============================================================================
# 3. CONTEXT RELEVANCE / RETRIEVAL PRECISION EVALUATOR
# ==============================================================================
class ContextRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain whether the retrieved facts contain information directly useful for answering the question."]
    relevant: Annotated[bool, ..., "True if retrieved facts contain relevant information; False if completely irrelevant noise."]

context_relevance_prompt = """You are evaluating retrieval quality in a RAG system.
You are given:
1. QUESTION
2. RETRIEVED FACTS

Grading Criteria:
- Check if the RETRIEVED FACTS contain key concepts, entities, or explanations needed to address the QUESTION.
- If the facts are completely unrelated noise -> Mark False.
- If the facts contain partial or full information relevant to the question -> Mark True."""

context_relevance_judge = judge_llm.with_structured_output(ContextRelevanceGrade, method="json_schema", strict=True)

def eval_context_relevance(inputs: dict, outputs: dict) -> dict:
    """Evaluates if the retrieved chunks contain relevant information for the question."""
    docs_text = "\n\n".join(doc.page_content for doc in outputs.get("documents", []))
    prompt = f"QUESTION:\n{inputs.get('question', '')}\n\nRETRIEVED FACTS:\n{docs_text}"
    result = context_relevance_judge.invoke([
        {"role": "system", "content": context_relevance_prompt},
        {"role": "user", "content": prompt}
    ])
    return {"key": "context_relevance", "score": int(result["relevant"]), "comment": result["explanation"]}

# ==============================================================================
# 4. ANSWER CORRECTNESS EVALUATOR (Ground Truth Accuracy)
# ==============================================================================
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Compare the student answer with the ground-truth reference answer step-by-step."]
    correct: Annotated[bool, ..., "True if the student answer is factually consistent and equivalent to the ground truth; False otherwise."]

correctness_prompt = """You are an expert professor evaluating answer factual accuracy.
You are given:
1. QUESTION
2. GROUND TRUTH (Reference Answer)
3. STUDENT ANSWER (Candidate Generated Response)

Grading Criteria:
- Focus strictly on factual accuracy and semantic equivalence.
- Paraphrasing or varying sentence structures MUST be graded as Correct as long as the facts match.
- If the student answer contradicts the ground truth or omits essential factual claims -> Mark False."""

correctness_judge = judge_llm.with_structured_output(CorrectnessGrade, method="json_schema", strict=True)

def eval_correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Evaluates semantic correctness against the human ground-truth answer."""
    prompt = (
        f"QUESTION: {inputs.get('question', '')}\n"
        f"GROUND TRUTH: {reference_outputs.get('answer', '')}\n"
        f"STUDENT ANSWER: {outputs.get('answer', '')}"
    )
    result = correctness_judge.invoke([
        {"role": "system", "content": correctness_prompt},
        {"role": "user", "content": prompt}
    ])
    return {"key": "correctness", "score": int(result["correct"]), "comment": result["explanation"]}

# ==============================================================================
# 5. REFUSAL & CONCISION EVALUATOR
# ==============================================================================
def eval_refusal_and_concision(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Deterministic check for concision and refusal consistency."""
    pred = outputs.get("answer", "").strip().lower()
    ref = reference_outputs.get("answer", "").strip().lower()
    
    is_ref_refusal = "do not have sufficient information" in ref
    if is_ref_refusal:
        score = 1.0 if "do not have sufficient information" in pred or "insufficient" in pred else 0.0
        comment = "Refusal check passed." if score == 1.0 else "Model failed to refuse an unanswerable query!"
    else:
        is_concise = len(pred) < (3.5 * len(ref) + 100)
        score = 1.0 if is_concise else 0.0
        comment = "Concision check passed." if is_concise else "Model was excessively verbose."
        
    return {"key": "refusal_and_concision", "score": int(score), "comment": comment}

print("✅ All 5 Enterprise RAG Evaluators initialized with Structured Output schemas.")


✅ All 5 Enterprise RAG Evaluators initialized with Structured Output schemas.


---
# 🧠 Module 7: Advanced LLM-as-a-Judge Paradigms & Bias Mitigation

### 🔍 What is happening here?
Using an LLM as a judge introduces specific, well-documented psychological and statistical biases:
1. **Position Bias**: In pairwise comparisons (e.g., Model A vs Model B), LLMs consistently favor whichever response is presented first (First-Position Bias) or last.
2. **Verbosity Bias**: LLM judges systematically assign higher ratings to wordier, longer, more decorative answers—even when the shorter answer is more concise and accurate!
3. **Self-Enhancement Bias**: A model (e.g., GPT-4) tends to rate outputs generated by itself or its own family higher than outputs from other model families (e.g., Claude, Llama, Gemini).
4. **Harshness / Strictness Drift**: Temperature fluctuations cause variance in grading strictness across runs.

### 💡 How to Mitigate LLM Judge Biases:
- **Technique 1: Explicit Anchor Rubrics (1-to-5 Likert Scale)**:
  Instead of asking the model "Rate from 1 to 5" (which is vague), provide explicit, objective behavioral anchors for *every single score point*.
- **Technique 2: Pairwise Swapping**:
  When comparing two models, run the evaluation twice with positions swapped ($A$ vs $B$, then $B$ vs $A$) and average the scores.
- **Technique 3: Fixed Temperature (0.0)**:
  Always evaluate at zero temperature to eliminate stochastic sampling drift.

### 🎯 Interview Insight:
> **Question**: *How do you prove that your LLM-as-a-judge is reliable and not just hallucinating its evaluations?*  
> **Answer**: You measure **Human-Judge Correlation**! Compute Cohen's Kappa (for categorical grades) or Pearson/Spearman Rank Correlation (for continuous scores) between human expert grades and LLM grades across a sampled validation set. High-performing calibrated judges achieve Spearman $\rho > 0.85$. Additionally, few-shot calibrated examples (giving the judge 2-3 anchor examples of a 1, 3, and 5) dramatically reduce rubric variance.


In [37]:
from pydantic import BaseModel, Field

class GradedRubricScore(BaseModel):
    chain_of_thought: str = Field(description="Detailed step-by-step reasoning evaluating the answer against each point of the rubric.")
    score: int = Field(ge=1, le=5, description="Integer score strictly between 1 and 5 according to the rubric.")
    rationale: str = Field(description="Summary sentence justifying the final assigned score.")

rubric_scoring_prompt = """You are an impartial academic evaluator applying a strict 5-point Likert scale.

EVALUATION RUBRIC:
- 1 Point (Severely Flawed): The answer is completely incorrect, irrelevant, or entirely fabricated (hallucinated).
- 2 Points (Poor): The answer mentions some relevant keywords, but contains major factual inaccuracies or misses the core question.
- 3 Points (Mediocre): The answer is partially correct and grounded, but omits critical nuances or contains minor irrelevant padding.
- 4 Points (Good): The answer is fully factually accurate, grounded in context, and addresses the question clearly with minor stylistic imperfections.
- 5 Points (Exemplary): The answer is perfectly accurate, concise, thoroughly grounded, and directly addresses the prompt without any fluff or omission.

Analyze the question, context, and student response step-by-step. Emit your chain of thought first, then the score."""

if os.getenv("OPENROUTER_API_KEY"):
    rubric_judge = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model="openai/gpt-4o-mini",
        temperature=0
    ).with_structured_output(GradedRubricScore)
else:
    rubric_judge = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(GradedRubricScore)

demo_q = "How does ReAct use self-reflection?"
demo_good_a = "ReAct interleaves reasoning and action traces, allowing the agent to observe tool outputs and adjust its trajectory."
demo_bad_a = "ReAct is a React.js frontend library for web development created by Facebook."

print("🧪 TESTING 5-POINT ANCHOR RUBRIC ON CONTRASTING ANSWERS:")
eval_good = rubric_judge.invoke([
    {"role": "system", "content": rubric_scoring_prompt},
    {"role": "user", "content": f"QUESTION: {demo_q}\nSTUDENT ANSWER: {demo_good_a}"}
])
print(f"\nGood Answer Score: {eval_good.score}/5")
print(f"Reasoning: {eval_good.chain_of_thought[:150]}...")

eval_bad = rubric_judge.invoke([
    {"role": "system", "content": rubric_scoring_prompt},
    {"role": "user", "content": f"QUESTION: {demo_q}\nSTUDENT ANSWER: {demo_bad_a}"}
])
print(f"\nBad Answer Score : {eval_bad.score}/5")
print(f"Reasoning: {eval_bad.chain_of_thought[:150]}...")


🧪 TESTING 5-POINT ANCHOR RUBRIC ON CONTRASTING ANSWERS:


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a18903-78f6-72d0-9c86-6828ffda5d86,id=01a18903-78f6-72d0-9c86-6828ffda5d86; trace=01a18903-78f6-72d0-9c86-6828ffda5d86,id=01a18903-78ff-7bd3-afe9-db22b90c9f6f



Good Answer Score: 3/5
Reasoning: The student's answer mentions that ReAct interleaves reasoning and action traces, which is relevant to the concept of self-reflection in the context o...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a18903-78f6-72d0-9c86-6828ffda5d86,id=01a18903-78ff-7bd3-afe9-db22b90c9f6f; trace=01a18903-78f6-72d0-9c86-6828ffda5d86,id=01a18903-902c-7a63-ab75-6fea0ecd8eb7; trace=01a18903-78f6-72d0-9c86-6828ffda5d86,id=01a18903-902c-7a63-ab75-6fea0ecd8eb7; trace=01a18903-78f6-72d0-9c86-6828ffda5d86,id=01a18903-78f6-72d0-9c86-6828ffda5d86; trace=01a18903-9071-7531-bbe2-891c8f5dcd33,id=01a18903-9071-7531-bbe2-891c8f5dcd33; trace=01a18903-9071-7531-bbe2-891c8f5dcd33,id=01a18903-9071-7531-bbe2-892306205ad8



Bad Answer Score : 1/5
Reasoning: The student's response does not address the question about how ReAct uses self-reflection. Instead, it provides information about ReAct being a React....


---
# 🏢 Module 8: Industry Frameworks Matrix (Ragas vs TruLens vs LangSmith vs DeepEval)

### 🔍 What is happening here?
Every modern enterprise uses one or more specialized frameworks for evaluation. Below is the definitive comparison matrix of the market leaders:

| Feature / Metric | **Ragas** | **TruLens** | **LangSmith** | **DeepEval / G-Eval** |
| :--- | :--- | :--- | :--- | :--- |
| **Core Philosophy** | Reference-free & Reference-based RAG metrics | The RAG Triad with feedback functions | End-to-end tracing, evaluation datasets, and production monitoring | Unit testing for LLMs with CI/CD integration |
| **Key Retrieval Metrics** | Context Precision, Context Recall, Context Entities Recall | Context Relevance | Retrieval Relevance, Custom Hit Rate/MRR | Contextual Precision, Contextual Recall |
| **Key Generation Metrics** | Faithfulness, Answer Relevance, Semantic Similarity | Groundedness, Answer Relevance | Correctness, Groundedness, Concision | Faithfulness, Hallucination, G-Eval Rubric |
| **Synthetic Test Generation** | **Yes (Evol-Instruct engine)** | Limited | No (Dataset curation UI) | Yes (Synthetic query generation) |
| **Best Used For** | Research & offline pipeline benchmarking | Tracing LLM feedback loops & cost | **Production monitoring, tracing, and team evaluation dashboards** | GitHub Actions CI/CD automated test gates |

---

### 📐 Ragas Mathematical Formulas Decoded:

#### 1. Context Precision in Ragas
Evaluates whether the chunks that contain ground-truth answers are ranked at the top of the context:
$$\text{Context Precision@K} = \frac{\sum_{k=1}^K (\text{Precision@k} \times v_k)}{\text{Total Relevant Chunks in Top-K}}$$
*(Where $v_k \in \{0, 1\}$ indicates whether chunk $k$ is relevant).*

#### 2. Context Recall in Ragas
Measures the proportion of sentences in the human ground-truth answer that can be attributed to the retrieved context:
$$\text{Context Recall} = \frac{|\text{Sentences in Ground Truth supported by Context}|}{|\text{Total Sentences in Ground Truth}|}$$


---
# 🚀 Module 9: Running the Comprehensive Evaluation Suite & Pandas Dashboard

### 🔍 What is happening here?
We execute `client.evaluate(...)` in LangSmith across our golden test cases using our 5 custom evaluators.  
Then, we convert the results into a **Pandas DataFrame** and compute:
- Aggregate Pass Rate (%)
- Component Breakdown (Retriever vs Generator health)
- Root-cause diagnostics for any failed test cases.

### 🎯 Interview Insight:
> **Question**: *How do you transition an evaluation suite from offline development into online production monitoring?*  
> **Answer**:  
> - **In Offline Development**: You run all evaluators (including reference-based metrics like Correctness and Context Recall) against a curated Golden Dataset on every git pull request.  
> - **In Online Production**: You don't have human reference answers for live user queries! Therefore, you sample a fraction (e.g., 5-10%) of production traffic and run **Reference-Free Evaluators** (Faithfulness, Answer Relevance, and Toxicity). Alerts are triggered if average Faithfulness drops below a target threshold (e.g., 95%).


In [38]:
# Define target execution wrapper for LangSmith evaluation runner
def evaluation_target(inputs: dict) -> dict:
    """Runs the RAG bot and packages output for evaluators."""
    question = inputs["question"]
    result = rag_bot(question)
    return {
        "answer": result["answer"],
        "documents": result["documents"]
    }

print("🚀 Launching Comprehensive RAG Evaluation Experiment via LangSmith...")
experiment_results = client.evaluate(
    evaluation_target,
    data=dataset_name,
    evaluators=[
        eval_groundedness,
        eval_answer_relevance,
        eval_context_relevance,
        eval_correctness,
        eval_refusal_and_concision
    ],
    experiment_prefix="master-rag-benchmark",
    metadata={
        "chunk_size": 250,
        "chunk_overlap": 30,
        "provider": "OpenRouter" if os.getenv("OPENROUTER_API_KEY") else "OpenAI",
        "embedding_model": "nvidia/nemotron-3.5-embed-1b:free" if os.getenv("OPENROUTER_API_KEY") else "text-embedding-3-small",
        "generator_model": "openai/gpt-4o-mini" if os.getenv("OPENROUTER_API_KEY") else "gpt-4o-mini",
        "top_k": 5
    }
)

print("\n🎉 Benchmark execution complete!")


🚀 Launching Comprehensive RAG Evaluation Experiment via LangSmith...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a18903-9071-7531-bbe2-891c8f5dcd33,id=01a18903-9071-7531-bbe2-892306205ad8; trace=01a18903-9071-7531-bbe2-891c8f5dcd33,id=01a18903-9910-7eb2-b8f8-c71c3fc036e3; trace=01a18903-9071-7531-bbe2-891c8f5dcd33,id=01a18903-9910-7eb2-b8f8-c71c3fc036e3; trace=01a18903-9071-7531-bbe2-891c8f5dcd33,id=01a18903-9071-7531-bbe2-891c8f5dcd33
d:\Agentic-AI-Practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'master-rag-benchmark-c4673ec1' at:
https://smith.langchain.com/o/b8939b9c-5c07-4f7f-8896-5f357d31717b/datasets/fe26d847-d354-454f-9951-dc3753fa4376/compare?selectedSessions=1331696c-9c64-4628-9edc-79ccbf87d02e




5it [00:52, 10.41s/it]


🎉 Benchmark execution complete!


In [39]:
import pandas as pd

# Extract results locally into a Pandas DataFrame
try:
    df_results = experiment_results.to_pandas()
    print("📋 Evaluation Experiment Results Table:")
    display_cols = [c for c in df_results.columns if any(k in c for k in ["question", "feedback", "score", "answer"])]
    print(df_results.head(5))
except Exception as e:
    print(f"Note on Pandas conversion: {e}")
    print("Experiment successfully registered in your LangSmith dashboard!")


ModuleNotFoundError: No module named 'pandas'

---
# 🩺 Module 10: RAG Root-Cause Failure Diagnosis Decision Tree

When a RAG test case fails, consult this diagnostic matrix to pinpoint the exact failure mechanism:

```
                                  [ RAG Failure Detected ]
                                             |
             +-------------------------------+-------------------------------+
             |                                                               |
             v                                                               v
  [ Low Context Relevance ]                                      [ High Context Relevance ]
             |                                                               |
             v                                                               v
   RETRIEVAL FAILURE                                              Is Groundedness High?
   - Chunks do not contain answer.                                           |
   - Fixes:                                              +-------------------+-------------------+
     * Increase Top-K (e.g., 5 -> 10)                    |                                       |
     * Tune chunk size & overlap                         v                                       v
     * Upgrade embedding model                    [ High Groundedness ]                  [ Low Groundedness ]
     * Add BM25 Hybrid Search                            |                                       |
     * Implement Cross-Encoder Re-ranker                 v                                       v
                                              Is Answer Relevance High?               GENERATION HALLUCINATION
                                                         |                            - Model invented facts
                                      +------------------+------------------+           outside context.
                                      |                                     |         - Fixes:
                                      v                                     v           * Lower temperature (0)
                             [ High Relevance ]                     [ Low Relevance ]   * Strict system prompt
                                      |                                     |           * Chain-of-Thought
                                      v                                     v           * Upgrade to GPT-4o
                             CORRECT PIPELINE                       INSTRUCTION EVASION
                             - If answer still doesn't match         - Model rambles or ignores
                               reference, check for missing            direct question prompt.
                               nuance in knowledge base.
```

---

# 🏆 The AI Engineer Interview Preparation Master Guide (10 Critical Q&As)

### Q1: What is the RAG Triad, and why are its 3 legs decoupled?
> **Answer**: The RAG Triad consists of **Context Relevance** (Query $\leftrightarrow$ Context), **Groundedness/Faithfulness** (Context $\leftrightarrow$ Answer), and **Answer Relevance** (Query $\leftrightarrow$ Answer). They are decoupled because RAG involves two distinct failure domains: Information Retrieval and Natural Language Generation. Decoupling them allows engineers to isolate whether errors stem from the vector index or from LLM synthesis.

### Q2: How do you evaluate a RAG system in production without ground truth labels?
> **Answer**: Use **Reference-Free Evaluation**. By measuring the RAG Triad (Context Relevance, Faithfulness, and Answer Relevance), you only require the User Query, the Retrieved Chunks, and the Generated Answer—none of which require human ground-truth answers. Sampling live traces and running reference-free evaluators detects production hallucinations in real time.

### Q3: What is the difference between Context Precision and Context Recall?
> **Answer**:
> - **Context Precision**: Measures signal-to-noise ratio in retrieved chunks, weighted by rank. It answers: *"Are the relevant chunks concentrated at the top of the context window?"*
> - **Context Recall**: Measures completeness. It answers: *"Did the retriever fetch all the facts necessary to answer every aspect of the question?"*

### Q4: Why do traditional metrics like BLEU and ROUGE fail for generative RAG?
> **Answer**: BLEU and ROUGE rely on lexical n-gram overlap. They penalize valid semantic paraphrasing, rewarded length/verbosity, and fail catastrophically on negation (e.g., omitting "not" flips factual correctness 100%, yet ROUGE scores exceed 90%).

### Q5: How do you identify and mitigate LLM-as-a-judge biases?
> **Answer**: The major biases are Position Bias, Verbosity Bias, Self-Enhancement Bias, and Strictness Drift. Mitigate them by:
> 1. Enforcing structured output schemas with step-by-step reasoning generated *before* the score.
> 2. Using explicit 1-to-5 Likert anchor rubrics with concrete behavioral descriptions.
> 3. Swapping model positions in pairwise evaluations ($A/B$ and $B/A$).
> 4. Setting temperature to 0.0 and measuring Spearman correlation against human expert ratings.

### Q6: What are Hit Rate@K and MRR@K, and how do you use them to optimize chunk size and Top-K?
> **Answer**: Hit Rate@K is the fraction of queries where at least one ground-truth document appears in the top $K$ results. MRR@K is the mean reciprocal rank ($1/\text{rank}$) of the first hit. When testing chunk sizes (e.g., 128 vs 256 vs 512 tokens), choose the configuration that maximizes Hit Rate and MRR on an offline retrieval dataset before running LLM synthesis.

### Q7: How do you handle and evaluate negative / unanswerable test queries?
> **Answer**: Include out-of-domain queries in your evaluation benchmark whose answers cannot be deduced from the corpus. Evaluate whether the model triggers a standard graceful refusal ("I do not have sufficient information") rather than hallucinating plausible false facts.

### Q8: What is G-Eval, and how does it differ from simple prompt-based grading?
> **Answer**: G-Eval (Liu et al.) is an evaluation framework that uses LLMs with Chain-of-Thought (CoT) to generate evaluation steps based on a task description and criteria. It calculates final scores by taking a probability-weighted average of the score tokens from the LLM's output probabilities, yielding smoother continuous scores that correlate higher with human judgments.

### Q9: How do you balance the cost and latency of LLM evaluators in production?
> **Answer**: Running a full GPT-4 evaluation on 100% of production traffic is cost-prohibitive. Production architectures use **Tiered Evaluation**:
> 1. Run fast, deterministic heuristic checks (length, latency, regex refusal) on 100% of queries.
> 2. Sample 2-5% of traffic for asynchronous background LLM-as-a-judge evaluation.
> 3. Use smaller, fine-tuned judge models (e.g., Llama-3-8B-Instruct or Prometheus) for evaluation rather than flagship models.

### Q10: If context relevance is 95% but answer correctness is 40%, what are the 3 most likely root causes?
> **Answer**:
> 1. **Prompt Inadequacy / Weak Instruction Following**: The system prompt fails to instruct the model to synthesize multiple chunks, or the prompt is too complex.
> 2. **Lost in the Middle**: Relevant information is located in the middle chunks of a long context window where LLM attention is weakest.
> 3. **Model Reasoning Limitations**: The task requires multi-hop logical deduction that exceeds the reasoning capabilities of the selected generator model (e.g., using an ultra-small model instead of a reasoning/frontier model).
